# gRPC y GraphQL — Práctica

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/18_intro_a_apis/code/04_grpc_graphql.ipynb)

En este notebook exploraremos dos paradigmas de comunicación modernos:

- **GraphQL**: un lenguaje de consulta para APIs que permite al cliente solicitar exactamente los datos que necesita.
- **gRPC**: un framework de llamadas a procedimiento remoto basado en HTTP/2 y Protocol Buffers.

Compararemos sus payloads con REST tradicional.

## 1. Setup

In [ ]:
%pip install grpcio grpcio-tools requests matplotlib

In [ ]:
import requests
import json
import sys
import matplotlib.pyplot as plt

## 2. GraphQL — Query básica

GraphQL permite al cliente especificar **exactamente** qué campos necesita.
A diferencia de REST, donde el servidor define la estructura de la respuesta,
en GraphQL el cliente envía una consulta describiendo los datos requeridos.

Usaremos la API pública de países: `https://countries.trevorblades.com/graphql`

In [ ]:
# Endpoint GraphQL
GRAPHQL_URL = "https://countries.trevorblades.com/graphql"

# Query: obtener todos los países con su nombre, código, capital y continente
query_completa = """
{
  countries {
    code
    name
    capital
    continent {
      name
    }
    languages {
      name
    }
    currency
    emoji
  }
}
"""

# Enviar la query como POST (estándar en GraphQL)
response_completa = requests.post(
    GRAPHQL_URL,
    json={"query": query_completa}
)

datos = response_completa.json()
print(f"Status: {response_completa.status_code}")
print(f"Número de países: {len(datos['data']['countries'])}")
print(f"\nPrimer país:")
print(json.dumps(datos['data']['countries'][0], indent=2, ensure_ascii=False))

### ¿Cómo funciona la query?

La query GraphQL tiene una estructura jerárquica:

```graphql
{
  countries {        # Recurso raíz
    code             # Campo escalar
    name             # Campo escalar
    continent {      # Relación (objeto anidado)
      name
    }
  }
}
```

Observa que:
- No hay endpoints diferentes para cada recurso (como en REST)
- El cliente decide qué campos recibir
- Las relaciones se resuelven en una sola petición

## 3. GraphQL — Query con campos específicos

Una de las ventajas principales de GraphQL es evitar el **over-fetching**.
Veamos cómo cambia el tamaño de la respuesta al pedir menos campos.

In [ ]:
# Query mínima: solo nombre y código
query_minima = """
{
  countries {
    code
    name
  }
}
"""

response_minima = requests.post(
    GRAPHQL_URL,
    json={"query": query_minima}
)

# Query intermedia: nombre, capital y continente
query_media = """
{
  countries {
    code
    name
    capital
    continent {
      name
    }
  }
}
"""

response_media = requests.post(
    GRAPHQL_URL,
    json={"query": query_media}
)

# Comparar tamaños
size_minima = len(response_minima.content)
size_media = len(response_media.content)
size_completa = len(response_completa.content)

print(f"Query mínima (code, name):                    {size_minima:,} bytes")
print(f"Query media (code, name, capital, continent):  {size_media:,} bytes")
print(f"Query completa (todos los campos):             {size_completa:,} bytes")
print(f"\nReducción mínima vs completa: {(1 - size_minima/size_completa)*100:.1f}%")

### ¿Por qué importa?

En REST, el servidor devuelve **todos** los campos del recurso.
Si solo necesitas el nombre de un país, de todas formas recibes la capital, moneda, idiomas, etc.

Con GraphQL, el cliente controla exactamente qué datos recibe.
Esto es especialmente útil en:
- Aplicaciones móviles con ancho de banda limitado
- Dashboards que solo muestran ciertos campos
- Microservicios que necesitan datos específicos de otros servicios

## 4. GraphQL vs REST — Comparación de payload

Comparemos una llamada equivalente usando REST y GraphQL.
Usaremos la API REST de `restcountries.com` para obtener los mismos datos.

In [ ]:
# REST: obtener todos los países (devuelve TODOS los campos)
rest_response = requests.get("https://restcountries.com/v3.1/all?fields=name,cca2,capital,continents,languages,currencies")
rest_size = len(rest_response.content)

# GraphQL: obtener datos equivalentes
graphql_equiv = """
{
  countries {
    code
    name
    capital
    continent {
      name
    }
    languages {
      name
    }
    currency
  }
}
"""
graphql_response = requests.post(GRAPHQL_URL, json={"query": graphql_equiv})
graphql_size = len(graphql_response.content)

print(f"REST (restcountries.com, campos filtrados): {rest_size:,} bytes")
print(f"GraphQL (countries API, campos equivalentes): {graphql_size:,} bytes")
print(f"\nNota: las APIs tienen estructuras de datos diferentes,")
print(f"por lo que la comparación no es 1:1, pero ilustra la diferencia.")

In [ ]:
# Comparación más justa: mismo dato, solo código y nombre
rest_minimal = requests.get("https://restcountries.com/v3.1/all?fields=name,cca2")
rest_minimal_size = len(rest_minimal.content)

graphql_minimal = requests.post(GRAPHQL_URL, json={"query": query_minima})
graphql_minimal_size = len(graphql_minimal.content)

print(f"Solo código y nombre:")
print(f"  REST:    {rest_minimal_size:,} bytes")
print(f"  GraphQL: {graphql_minimal_size:,} bytes")
print(f"\nMuestra de REST:")
print(json.dumps(rest_minimal.json()[:2], indent=2, ensure_ascii=False))
print(f"\nMuestra de GraphQL:")
print(json.dumps(graphql_minimal.json()['data']['countries'][:2], indent=2, ensure_ascii=False))

## 5. gRPC — Conceptual

gRPC usa **Protocol Buffers** (protobuf) como formato de serialización y **HTTP/2** como transporte.

### ¿Qué es un archivo `.proto`?

Define el contrato entre cliente y servidor: los mensajes (datos) y los servicios (operaciones).

In [ ]:
# Definición de un servicio gRPC en Protocol Buffers
proto_definition = '''
syntax = "proto3";

package countries;

// Mensaje que representa un país
message Country {
  string code = 1;        // Campo 1: código del país
  string name = 2;        // Campo 2: nombre
  string capital = 3;     // Campo 3: capital
  string continent = 4;   // Campo 4: continente
  repeated string languages = 5;  // Campo 5: lista de idiomas
  string currency = 6;    // Campo 6: moneda
}

// Mensaje de solicitud
message CountryRequest {
  string code = 1;  // Código del país a buscar
}

// Mensaje de respuesta con lista de países
message CountryList {
  repeated Country countries = 1;
}

// Definición del servicio
service CountryService {
  // Obtener un país por código
  rpc GetCountry (CountryRequest) returns (Country);
  // Obtener todos los países
  rpc ListCountries (google.protobuf.Empty) returns (CountryList);
  // Stream de países (server streaming)
  rpc StreamCountries (google.protobuf.Empty) returns (stream Country);
}
'''

print(proto_definition)

### Compilación del `.proto`

El compilador `protoc` genera código Python automáticamente:

```bash
python -m grpc_tools.protoc \
  -I. \
  --python_out=. \
  --grpc_python_out=. \
  countries.proto
```

Esto genera dos archivos:
- `countries_pb2.py` — clases para los mensajes (Country, CountryRequest, etc.)
- `countries_pb2_grpc.py` — stubs del cliente y servidor

In [ ]:
# Simulación: cómo se vería usar el cliente generado
# (no podemos ejecutar esto sin un servidor gRPC real)

codigo_cliente_grpc = '''
import grpc
import countries_pb2
import countries_pb2_grpc

# Crear canal de comunicación
channel = grpc.insecure_channel('localhost:50051')

# Crear stub (cliente)
stub = countries_pb2_grpc.CountryServiceStub(channel)

# Llamar al método GetCountry
request = countries_pb2.CountryRequest(code="MX")
response = stub.GetCountry(request)

print(f"País: {response.name}")
print(f"Capital: {response.capital}")
print(f"Continente: {response.continent}")

# Server streaming: recibir países uno por uno
for country in stub.StreamCountries(google.protobuf.Empty()):
    print(f"{country.code}: {country.name}")
'''

print("Código del cliente gRPC (conceptual):")
print(codigo_cliente_grpc)

### Protobuf: serialización binaria

Podemos demostrar cómo protobuf serializa datos de forma compacta
sin necesitar un servidor gRPC.

In [ ]:
import struct

# Simulación de cómo protobuf codifica un país
# En protobuf, cada campo se identifica por número, no por nombre

pais_json = {
    "code": "MX",
    "name": "Mexico",
    "capital": "Mexico City",
    "continent": "North America",
    "languages": ["Spanish"],
    "currency": "MXN"
}

# Tamaño en JSON
json_bytes = json.dumps(pais_json).encode('utf-8')
print(f"JSON: {len(json_bytes)} bytes")
print(f"Contenido: {json_bytes.decode()}")
print()

# Simulación simplificada de codificación protobuf
# En protobuf real, los campos se codifican como: (field_number << 3 | wire_type) + valor
# wire_type 2 = length-delimited (strings, bytes, mensajes anidados)

def encode_string_field(field_number, value):
    """Codifica un campo string en formato protobuf simplificado."""
    tag = (field_number << 3) | 2  # wire type 2 = length-delimited
    encoded_value = value.encode('utf-8')
    # tag (varint) + longitud (varint) + datos
    return bytes([tag, len(encoded_value)]) + encoded_value

# Codificar el país en "protobuf" simplificado
proto_bytes = b''
proto_bytes += encode_string_field(1, "MX")             # code
proto_bytes += encode_string_field(2, "Mexico")          # name
proto_bytes += encode_string_field(3, "Mexico City")     # capital
proto_bytes += encode_string_field(4, "North America")   # continent
proto_bytes += encode_string_field(5, "Spanish")         # languages[0]
proto_bytes += encode_string_field(6, "MXN")             # currency

print(f"Protobuf (simplificado): {len(proto_bytes)} bytes")
print(f"Contenido (hex): {proto_bytes.hex()}")
print()
print(f"Reducción: {(1 - len(proto_bytes)/len(json_bytes))*100:.1f}%")
print()
print("Nota: el JSON incluye nombres de campo como 'code', 'name', etc.")
print("Protobuf usa números de campo (1, 2, 3...) que ocupan 1 byte cada uno.")

### ¿Por qué protobuf es más compacto?

| Aspecto | JSON | Protobuf |
|---------|------|----------|
| Nombres de campo | Strings completos (`"capital"`) | Números de 1 byte (`3`) |
| Delimitadores | `{`, `}`, `:`, `,`, `"` | No necesita |
| Formato | Texto legible | Binario |
| Tipos | Todo es string | Tipado nativo |

El costo: protobuf **no es legible** sin el `.proto` que define la estructura.

## 6. Comparación de tamaño de payload

Resumen visual de los tamaños de payload para representar un país.

In [ ]:
# Datos de un país en diferentes formatos
pais_data = {
    "code": "MX",
    "name": "Mexico",
    "capital": "Mexico City",
    "continent": "North America",
    "languages": ["Spanish"],
    "currency": "MXN"
}

# REST JSON (con nombres de campo completos)
rest_json = json.dumps(pais_data)
rest_json_size = len(rest_json.encode('utf-8'))

# GraphQL JSON (misma estructura pero sin over-fetching)
# En la práctica es similar al REST si pedimos los mismos campos,
# pero incluye el wrapper {"data": {"country": ...}}
graphql_json = json.dumps({"data": {"country": pais_data}})
graphql_json_size = len(graphql_json.encode('utf-8'))

# Protobuf (estimado basado en nuestra codificación simplificada)
protobuf_size = len(proto_bytes)

# SOAP XML (estimado)
soap_xml = f"""<?xml version="1.0"?>
<soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <soap:Body>
    <GetCountryResponse xmlns="http://example.com/countries">
      <Country>
        <Code>MX</Code>
        <Name>Mexico</Name>
        <Capital>Mexico City</Capital>
        <Continent>North America</Continent>
        <Languages>
          <Language>Spanish</Language>
        </Languages>
        <Currency>MXN</Currency>
      </Country>
    </GetCountryResponse>
  </soap:Body>
</soap:Envelope>"""
soap_size = len(soap_xml.encode('utf-8'))

print(f"Tamaño de payload para representar un país:")
print(f"  REST JSON:     {rest_json_size:>4} bytes")
print(f"  GraphQL JSON:  {graphql_json_size:>4} bytes")
print(f"  Protobuf:      {protobuf_size:>4} bytes")
print(f"  SOAP XML:      {soap_size:>4} bytes")

In [ ]:
# Gráfica de comparación
formatos = ['SOAP\n(XML)', 'REST\n(JSON)', 'GraphQL\n(JSON)', 'gRPC\n(Protobuf)']
sizes = [soap_size, rest_json_size, graphql_json_size, protobuf_size]
colores = ['#e74c3c', '#3498db', '#9b59b6', '#2ecc71']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(formatos, sizes, color=colores, edgecolor='white', linewidth=1.5)

# Agregar etiquetas de valor
for bar, size in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{size} B', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Tama\u00f1o (bytes)', fontsize=12)
ax.set_title('Comparaci\u00f3n de tama\u00f1o de payload por protocolo\n(1 registro de pa\u00eds)', fontsize=14)
ax.set_ylim(0, max(sizes) * 1.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

### ¿Por qué estos resultados?

- **SOAP** es el más grande porque XML es verboso: cada dato requiere etiquetas de apertura y cierre, más el envelope SOAP obligatorio.
- **REST** y **GraphQL** usan JSON, que es más compacto que XML pero todavía incluye nombres de campo como strings y delimitadores.
- **Protobuf** es el más compacto porque usa codificación binaria: los campos se identifican por números (1 byte) en lugar de nombres, y no necesita delimitadores de texto.

En la práctica, la diferencia se amplifica con volúmenes grandes de datos.

## Resumen

| Característica | REST | GraphQL | gRPC |
|----------------|------|---------|------|
| Formato | JSON (texto) | JSON (texto) | Protobuf (binario) |
| Transporte | HTTP/1.1 | HTTP/1.1 | HTTP/2 |
| Contrato | OpenAPI/Swagger | Schema GraphQL | `.proto` |
| Over-fetching | Sí | No | No |
| Streaming | Limitado | Subscriptions | Nativo (bidireccional) |
| Facilidad de uso | Alta | Media | Baja (requiere compilación) |
| Tamaño de payload | Mediano | Mediano (controlable) | Pequeño |
| Ideal para | APIs públicas | Frontends flexibles | Microservicios internos |